# Interactive Throw Prediction Heatmap

Drag sliders to move the thrower position and see how throw predictions change!

In [33]:
# %pip install ipywidgets

In [34]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import joblib
from ipywidgets import interact, widgets, Layout
from IPython.display import display
import psycopg2
from psycopg2.extras import RealDictCursor

from nflows import flows, distributions, transforms
from nflows.nn import nets

print("✅ Imports loaded")

✅ Imports loaded


## 1. Load Trained MDN Model

In [35]:
# Define model classes (must match training)

class ContextNetwork(nn.Module):
    """Processes context (player + position) before feeding to flow."""
    def __init__(self, n_players, embedding_dim=16, hidden_dim=64, output_dim=32):
        super().__init__()
        self.player_embedding = nn.Embedding(n_players, embedding_dim)
        input_dim = embedding_dim + 2
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, context):
        player_ids = context[:, 0].long()
        position = context[:, 1:3]
        player_emb = self.player_embedding(player_ids)
        combined = torch.cat([player_emb, position], dim=1)
        return self.network(combined)


def create_flow(n_players, num_layers=5, hidden_features=128, context_features=32):
    """Recreate flow architecture for loading weights."""
    base_dist = distributions.StandardNormal(shape=[2])
    
    context_net = ContextNetwork(
        n_players=n_players,
        embedding_dim=16,
        hidden_dim=64,
        output_dim=context_features
    )
    
    transform_list = []
    for _ in range(num_layers):
        transform_list.append(
            transforms.MaskedAffineAutoregressiveTransform(
                features=2,
                hidden_features=hidden_features,
                context_features=context_features,
                num_blocks=2
            )
        )
        transform_list.append(transforms.ReversePermutation(features=2))
    
    transform = transforms.CompositeTransform(transform_list)
    flow = flows.Flow(transform, base_dist)
    
    return flow, context_net

print("✅ Model classes defined")

✅ Model classes defined


In [36]:
# Load saved normalizing flow model
model_path = 'models/normalizing_flow_model.pkl'
save_dict = joblib.load(model_path)

hyperparams = save_dict['hyperparameters']
player_encoder = save_dict['player_encoder']
n_players = hyperparams['n_players']

# Recreate architecture and load weights
flow, context_net = create_flow(
    n_players=n_players,
    num_layers=hyperparams['num_layers'],
    hidden_features=hyperparams['hidden_features'],
    context_features=hyperparams['context_features']
)

flow.load_state_dict(save_dict['flow_state_dict'])
context_net.load_state_dict(save_dict['context_net_state_dict'])

flow.eval()
context_net.eval()

print(f"✅ Normalizing Flow model loaded!")
print(f"   Players: {n_players}")
print(f"   Flow layers: {hyperparams['num_layers']}")
print(f"   Best val loss: {save_dict['best_val_loss']:.4f}")

✅ Normalizing Flow model loaded!
   Players: 244
   Flow layers: 5
   Best val loss: -1.4279


## 2. Get Top Players

In [37]:
# Query ALL players with 500+ throws
DB_CONFIG = {
    'dbname': 'ufa_analytics',
    'user': 'joemolder',
    'password': '',
    'host': 'localhost',
    'port': 5432
}

conn = psycopg2.connect(**DB_CONFIG, cursor_factory=RealDictCursor)
cur = conn.cursor()

cur.execute("""
    SELECT 
        thrower as player_id,
        COUNT(*) as throw_count
    FROM events
    WHERE thrower IS NOT NULL
        AND event_type IN (18, 19, 20, 22)
    GROUP BY thrower
    HAVING COUNT(*) >= 500
    ORDER BY COUNT(*) DESC;
""")

all_players = cur.fetchall()
cur.close()
conn.close()

# Filter to players in the model's encoder
player_names = [p['player_id'] for p in all_players if p['player_id'] in player_encoder.classes_]
player_throw_counts = {p['player_id']: p['throw_count'] for p in all_players if p['player_id'] in player_encoder.classes_}

print(f"✅ Loaded {len(player_names)} players with 500+ throws")
print(f"   Top 5: {player_names[:5]}")
print(f"   Bottom 5: {player_names[-5:]}")

✅ Loaded 244 players with 500+ throws
   Top 5: ['pjanas', 'aroy', 'bsadok', 'ataylor', 'srueschem']
   Bottom 5: ['wlewis', 'kdelorenz', 'nransom', 'tdavis', 'bpastor']


## 3. Prediction Function

In [38]:
def predict_throw_distribution(player_name, thrower_x, thrower_y, grid_size=200):
    """
    Predict throw distribution using normalizing flow.
    Evaluates exact log-probability at each grid point.
    
    Returns:
        grid: [grid_y, grid_x] probability distribution
    """
    # Encode player
    try:
        player_encoded = player_encoder.transform([player_name])[0]
    except:
        raise ValueError(f"Player '{player_name}' not in training data")
    
    # Normalize coordinates
    thrower_x_norm = (thrower_x + 25) / 50
    thrower_y_norm = thrower_y / 120
    
    # Create context tensor
    context_input = torch.FloatTensor([[player_encoded, thrower_x_norm, thrower_y_norm]])
    
    with torch.no_grad():
        # Process context once
        context_features = context_net(context_input)
        
        # Create grid points
        x_bins = np.linspace(0, 1, grid_size)
        y_bins = np.linspace(0, 1, int(grid_size * 1.2))
        
        # Vectorized: evaluate all grid points at once
        xx, yy = np.meshgrid(x_bins, y_bins)
        grid_points = torch.FloatTensor(
            np.stack([xx.ravel(), yy.ravel()], axis=1)
        )  # [grid_size * grid_size, 2]
        
        # Expand context to match grid points
        context_expanded = context_features.expand(grid_points.shape[0], -1)
        
        # Compute log-probabilities for all points at once
        log_probs = flow.log_prob(grid_points, context=context_expanded)
        probs = torch.exp(log_probs).numpy()
        
        # Reshape to grid
        grid = probs.reshape(len(y_bins), len(x_bins))
    
    return grid

## 4. Interactive Visualization

Use the sliders and dropdown to explore throw predictions!

In [39]:
def plot_heatmap(player_name, thrower_x, thrower_y):
    """Plot interactive heatmap using normalizing flow."""
    try:
        grid = predict_throw_distribution(player_name, thrower_x, thrower_y)
    except Exception as e:
        print(f"Error: {e}")
        return
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Heatmap
    extent = [-25, 25, 0, 120]
    im = ax.imshow(grid, extent=extent, origin='lower', cmap='hot', alpha=0.8)
    
    # Thrower position
    ax.plot(thrower_x, thrower_y, 'o', color='cyan', markersize=20, 
            markeredgecolor='white', markeredgewidth=3, label='Thrower', zorder=10)
    
    # Field markings
    ax.axhline(y=20, color='white', linestyle='--', alpha=0.6, linewidth=2, label='Goal line')
    ax.axhline(y=100, color='white', linestyle='--', alpha=0.6, linewidth=2)
    ax.axhline(y=60, color='white', linestyle=':', alpha=0.4, linewidth=2, label='Midfield')
    
    # Field boundaries
    ax.axvline(x=-25, color='white', linewidth=3)
    ax.axvline(x=25, color='white', linewidth=3)
    ax.axhline(y=0, color='white', linewidth=3)
    ax.axhline(y=120, color='white', linewidth=3)
    
    # Endzones (shaded)
    ax.axhspan(0, 20, alpha=0.1, color='yellow', zorder=0)
    ax.axhspan(100, 120, alpha=0.1, color='yellow', zorder=0)
    
    ax.set_xlim(-27, 27)
    ax.set_ylim(0, 120)
    ax.set_xlabel('Field X (meters)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Field Y (meters)', fontsize=14, fontweight='bold')
    ax.set_title(f'{player_name} - Throw Prediction from ({thrower_x:.1f}, {thrower_y:.1f})',
                 fontsize=16, fontweight='bold', pad=20)
    ax.legend(loc='upper right', fontsize=11, framealpha=0.9)
    ax.set_facecolor('#1a4d2e')  # Field green
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, label='Probability Density', fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=11)
    
    plt.tight_layout()
    plt.show()
    
    # Print stats
    throw_count = player_throw_counts.get(player_name, 'N/A')
    print(f"\n{player_name} | Throws: {throw_count} | Position: ({thrower_x:.1f}, {thrower_y:.1f})")
    print(f"Peak density: {grid.max():.4f}")

In [ ]:
# Searchable player selector using Combobox
player_search = widgets.Combobox(
    placeholder='Type to search players...',
    options=player_names,
    value=player_names[0],
    description='Player:',
    ensure_option=True,  # Must match a valid player
    style={'description_width': '80px'},
    layout=Layout(width='400px')
)

x_slider = widgets.FloatSlider(
    value=0,
    min=-25,
    max=25,
    step=1,
    description='Field X:',
    continuous_update=True,
    style={'description_width': '80px'},
    layout=Layout(width='600px'),
    readout_format='.1f'
)

y_slider = widgets.FloatSlider(
    value=60,
    min=0,
    max=120,
    step=2,
    description='Field Y:',
    continuous_update=True,
    style={'description_width': '80px'},
    layout=Layout(width='600px'),
    readout_format='.1f'
)

# Interactive plot
interact(plot_heatmap, 
         player_name=player_search,
         thrower_x=x_slider,
         thrower_y=y_slider)

interactive(children=(Combobox(value='pjanas', description='Player:', ensure_option=True, layout=Layout(width=…

<function __main__.plot_heatmap(player_name, thrower_x, thrower_y)>

## Tips for Exploration

**Try these scenarios:**

1. **Deep in own endzone** (X=0, Y=10)
   - Look for dump/reset options vs deep shots
   
2. **Midfield** (X=0, Y=60)
   - Should show balanced short/medium/long options
   
3. **Near goal line** (X=0, Y=95)
   - Should favor scoring throws into endzone
   
4. **Sideline** (X=20, Y=60)
   - Look for cross-field throws vs continuation
   
5. **Compare different players**
   - Handlers vs cutters
   - Conservative vs aggressive throwers

## Summary

This interactive notebook lets you explore how throw predictions vary by:
- **Player** - Each player has unique throwing patterns
- **Position** - Field location dramatically affects throw options

The MDN model learns:
- 3 distinct throw modes (short/medium/long)
- Player-specific tendencies via embeddings
- Context-dependent strategies (field position)

**Next steps:**
- Build web app with smooth dragging (React + FastAPI)
- Add game context (score, time, O/D line)
- Compare with normalizing flow predictions